# Task 3 — Data Cleaning and Quality Assurance
**Tools:** Python, pandas, NumPy, Jupyter Notebook  
**Dataset:** Deliberately messy, synthetic Titanic-style passenger dataset  
**Objective:** Audit, clean, validate, document, and export an analysis-ready dataset.

> **Dataset transparency:** This project uses a reproducible synthetic passenger dataset inspired by the structure of the public Titanic training dataset. It is not presented as authentic historical passenger records. The source dataset for this exercise is generated with a fixed random seed and intentionally corrupted to test cleaning logic. If your instructor specifically requires a Kaggle-sourced dataset, replace the input CSV with the Kaggle Titanic `train.csv` and adapt the corruption step accordingly.

## 1. Cleaning policy and decision log

| Issue | Rule / treatment | Reason |
|---|---|---|
| Blank strings and whitespace | Convert to missing values | Blank-looking cells should be counted as missing, not valid strings. |
| Age | Parse numeric; values outside 0–100 become missing; impute median within passenger class and sex, then overall median | Age is skewed and group distributions differ; median is robust to extreme values. |
| Fare | Remove currency symbols/commas; parse numeric; values outside 0–10,000 become missing; impute class median, then overall median | Fare is right-skewed and depends strongly on class; class median is robust. |
| Sex | Trim/lowercase; map `m`/`f` and case variants to `Male`/`Female`; invalid categories become missing and are mode-imputed | Prevents category fragmentation and avoids inventing an unsupported category. |
| Embarked | Standardize known variants to `S`, `C`, `Q`; invalid/unrecognized values become missing and are mode-imputed | Uses a documented mapping to canonical codes. |
| Survived / Pclass | Coerce to numeric; values outside allowed categories become missing and mode-imputed | Values such as `yes`, `-1`, or class `4` violate the field definition. |
| SibSp / Parch | Coerce to numeric; values outside 0–20 become missing and mode-imputed | Counts cannot be negative; 20 is a conservative validation ceiling for this practice dataset. |
| Cabin | Retain information as `CabinKnown` (1 if present, else 0), then drop sparse text column | Cabin is mostly missing; imputing a cabin number would fabricate information. |
| Name / Ticket | Trim whitespace; missing values labelled `Unknown` | Keeps records while making unavailable text explicit. |
| Exact duplicate rows | Remove after normalization | Exact duplicates inflate counts and can bias summaries. Passenger IDs are kept as strings. |
| Numeric outliers | Detect with IQR; review against domain rules; retain plausible high fares, set impossible/invalid values to missing before imputation | Statistical outlier does not automatically mean an error. Do not indiscriminately delete valid passengers. |
| Row deletion | No rows deleted solely for missing predictors; exact duplicates removed | Preserves the sample size while handling recoverable missing values. |

## 2. Imports and load the messy dataset

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("data/titanic_style_messy_dataset.csv")
df_raw = pd.read_csv(DATA_PATH, dtype={"PassengerId": "string"})
df = df_raw.copy()

print("Shape:", df.shape)
display(df.head())
display(df.info())

## 3. Data quality report — before cleaning

In [ ]:
def data_quality_report(data):
    report = pd.DataFrame({
        "dtype": data.dtypes.astype(str),
        "null_count": data.isna().sum(),
        "null_percent": (data.isna().mean() * 100).round(2),
        "unique_values": data.nunique(dropna=True)
    })
    return report.sort_values(["null_count", "null_percent"], ascending=False)

quality_before = data_quality_report(df)
display(quality_before)
print("Rows:", len(df))
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Total missing cells:", int(df.isna().sum().sum()))
print("\nNumeric summary (coerce numeric-looking columns for diagnostics):")
display(df[["Age", "Fare", "Survived", "Pclass", "SibSp", "Parch"]].apply(pd.to_numeric, errors="coerce").describe().T)

### 3.1 Value-range and category anomaly checks

In [ ]:
numeric_for_audit = {
    "Age": (0, 100),
    "Fare": (0, 10000),
    "Survived": (0, 1),
    "Pclass": (1, 3),
    "SibSp": (0, 20),
    "Parch": (0, 20)
}
range_audit = []
for col, (low, high) in numeric_for_audit.items():
    values = pd.to_numeric(df[col].astype("string").str.replace(r"[$£,\s]", "", regex=True), errors="coerce")
    range_audit.append({
        "column": col,
        "min_observed": values.min(),
        "max_observed": values.max(),
        "outside_expected_range": int((~values.between(low, high) & values.notna()).sum()),
        "unparseable_non_null": int((values.isna() & df[col].notna()).sum())
    })
display(pd.DataFrame(range_audit))
print("Sex values before standardization:", df["Sex"].value_counts(dropna=False).head(20))
print("Embarked values before standardization:", df["Embarked"].value_counts(dropna=False).head(20))

## 4. Missing-value and standardization decisions
The transformation below first normalizes blank strings and parses fields, then applies domain validation. Imputation is performed only after invalid values have been converted to missing. This prevents malformed strings or impossible values from influencing medians and modes.

## 5. Cleaning pipeline

In [ ]:
# Normalize blanks / whitespace
clean = df.copy().replace(r"^\s*$", np.nan, regex=True)

# IDs are identifiers, not measures
clean["PassengerId"] = pd.to_numeric(clean["PassengerId"], errors="coerce").astype("Int64").astype("string")

# Parse numeric columns, including currency-formatted fare values
clean["Age"] = pd.to_numeric(
    clean["Age"].astype("string").str.strip().str.replace(r"[^0-9.\-]", "", regex=True),
    errors="coerce"
)
clean["Fare"] = pd.to_numeric(
    clean["Fare"].astype("string").str.strip().str.replace(r"[$£,\s]", "", regex=True),
    errors="coerce"
)
for col in ["Survived", "Pclass", "SibSp", "Parch"]:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

# Domain validation: impossible values become missing, not silently accepted
clean.loc[~clean["Age"].between(0, 100), "Age"] = np.nan
clean.loc[~clean["Fare"].between(0, 10000), "Fare"] = np.nan
clean.loc[~clean["Survived"].isin([0, 1]), "Survived"] = np.nan
clean.loc[~clean["Pclass"].isin([1, 2, 3]), "Pclass"] = np.nan
clean.loc[~clean["SibSp"].between(0, 20), "SibSp"] = np.nan
clean.loc[~clean["Parch"].between(0, 20), "Parch"] = np.nan

# Canonical category mappings
clean["Sex"] = clean["Sex"].astype("string").str.strip().str.lower()
clean["Sex"] = clean["Sex"].replace({"m": "male", "f": "female"})
clean.loc[~clean["Sex"].isin(["male", "female"]), "Sex"] = pd.NA
clean["Sex"] = clean["Sex"].str.title()

embarked_map = {
    "s": "S", "south": "S", "southampton": "S",
    "c": "C", "cherbourg": "C",
    "q": "Q", "queenstown": "Q"
}
clean["Embarked"] = clean["Embarked"].astype("string").str.strip().str.lower().map(embarked_map)

for col in ["Name", "Ticket", "Cabin"]:
    clean[col] = clean[col].astype("string").str.strip().replace("", pd.NA)

# Remove exact duplicate records; record the number explicitly
duplicates_removed = int(clean.duplicated().sum())
clean = clean.drop_duplicates().copy()

# Cabin sparsity: preserve whether a cabin value was available, not a fabricated cabin number
clean["CabinKnown"] = clean["Cabin"].notna().astype("int8")
clean = clean.drop(columns=["Cabin"])

# Robust imputation
clean["Age"] = clean["Age"].fillna(
    clean.groupby(["Pclass", "Sex"], dropna=False)["Age"].transform("median")
)
clean["Age"] = clean["Age"].fillna(clean["Age"].median())

clean["Fare"] = clean["Fare"].fillna(clean.groupby("Pclass")["Fare"].transform("median"))
clean["Fare"] = clean["Fare"].fillna(clean["Fare"].median())

for col in ["Sex", "Embarked", "Pclass", "Survived", "SibSp", "Parch"]:
    if clean[col].isna().any():
        mode_values = clean[col].mode(dropna=True)
        if len(mode_values):
            clean[col] = clean[col].fillna(mode_values.iloc[0])

for col in ["Name", "Ticket"]:
    clean[col] = clean[col].fillna("Unknown")

# Correct data types
clean["PassengerId"] = clean["PassengerId"].astype("string")
clean["Survived"] = clean["Survived"].astype("int8")
clean["Pclass"] = clean["Pclass"].astype("int8")
clean["Sex"] = clean["Sex"].astype("category")
clean["Age"] = clean["Age"].astype("float64").round(1)
clean["SibSp"] = clean["SibSp"].astype("int8")
clean["Parch"] = clean["Parch"].astype("int8")
clean["Ticket"] = clean["Ticket"].astype("string")
clean["Fare"] = clean["Fare"].astype("float64").round(2)
clean["CabinKnown"] = clean["CabinKnown"].astype("int8")
clean["Embarked"] = clean["Embarked"].astype("category")

clean = clean.reset_index(drop=True)
display(clean.head())
print("Duplicate rows removed:", duplicates_removed)
print("Cleaned shape:", clean.shape)

## 6. Outlier detection using the IQR method

In [ ]:
def iqr_outlier_report(data, columns):
    rows = []
    for col in columns:
        series = pd.to_numeric(data[col], errors="coerce").dropna()
        q1, q3 = series.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        count = int(((series < lower) | (series > upper)).sum())
        rows.append({
            "column": col, "Q1": q1, "Q3": q3, "IQR": iqr,
            "lower_fence": lower, "upper_fence": upper,
            "outlier_count": count,
            "decision": "Retain plausible values; domain-invalid values were set missing before imputation"
        })
    return pd.DataFrame(rows)

outlier_report = iqr_outlier_report(clean, ["Age", "Fare", "SibSp", "Parch"])
display(outlier_report)

### Outlier decision
IQR is used as a **screening tool**, not as an automatic deletion rule. A high fare can be legitimate (for example, a first-class fare), and deleting all IQR outliers would bias the dataset. Therefore, statistically unusual but domain-valid values are retained. Values outside documented domain bounds are treated as invalid/missing and imputed as described above. No rows are removed solely because they are IQR outliers.

## 7. Data quality report — after cleaning

In [ ]:
quality_after = data_quality_report(clean)
display(quality_after)
print("Rows:", len(clean))
print("Exact duplicate rows:", int(clean.duplicated().sum()))
print("Total missing cells:", int(clean.isna().sum().sum()))
print("\nFinal data types:")
display(clean.dtypes.astype(str).rename("dtype").to_frame())

## 8. Before-versus-after summary

In [ ]:
before_rows = len(df_raw)
after_rows = len(clean)
before_duplicate_count = int(df_raw.duplicated().sum())
after_duplicate_count = int(clean.duplicated().sum())
before_null_count = int(df_raw.isna().sum().sum())
after_null_count = int(clean.isna().sum().sum())

summary = pd.DataFrame([
    {"Metric": "Row count", "Before cleaning": before_rows, "After cleaning": after_rows},
    {"Metric": "Exact duplicate rows", "Before cleaning": before_duplicate_count, "After cleaning": after_duplicate_count},
    {"Metric": "Missing cells (explicit NaN)", "Before cleaning": before_null_count, "After cleaning": after_null_count},
    {"Metric": "Columns", "Before cleaning": df_raw.shape[1], "After cleaning": clean.shape[1]},
    {"Metric": "Columns with expected final dtype", "Before cleaning": "Audit required", "After cleaning": int(clean.dtypes.notna().sum())}
])
display(summary)
print("Rows removed as exact duplicates:", before_rows - after_rows)

## 9. Final validation checklist
The assertions below are intentionally strict. If any assertion fails, do not submit the output until the issue is investigated.

In [ ]:
assert clean.isna().sum().sum() == 0, "Missing values remain."
assert clean.duplicated().sum() == 0, "Duplicate rows remain."
assert clean["PassengerId"].notna().all(), "PassengerId contains missing values."
assert clean["Survived"].isin([0, 1]).all(), "Invalid survival labels remain."
assert clean["Pclass"].isin([1, 2, 3]).all(), "Invalid passenger classes remain."
assert clean["Age"].between(0, 100).all(), "Age outside expected range."
assert clean["Fare"].between(0, 10000).all(), "Fare outside expected range."
assert set(clean["Sex"].astype(str).unique()).issubset({"Male", "Female"}), "Unexpected Sex category."
assert set(clean["Embarked"].astype(str).unique()).issubset({"S", "C", "Q"}), "Unexpected Embarked category."
assert clean["PassengerId"].dtype.name in ["string", "object"], "ID should be string-like."
assert pd.api.types.is_float_dtype(clean["Fare"]), "Fare should be float."
assert pd.api.types.is_float_dtype(clean["Age"]), "Age should be float."
print("PASS: all final validation checks passed.")

## 10. Export the analysis-ready dataset

In [ ]:
OUTPUT_PATH = Path("data/titanic_style_cleaned_dataset.csv")
clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset to: {OUTPUT_PATH.resolve()}")
print("Final output shape:", clean.shape)

## 11. Conclusion
The workflow creates a repeatable audit trail: inspect the original data, identify missingness and range anomalies, normalize formatting, validate domains, remove exact duplicates, apply column-specific imputation, correct data types, review IQR outliers, run assertions, and export a separate clean CSV. The raw input remains unchanged so the process is auditable and reproducible.

**Important:** This is a synthetic training dataset. Do not describe it as real Titanic passenger data. If a real Kaggle dataset is mandatory, cite the Kaggle Titanic competition and use its downloaded `train.csv` as the input dataset.